In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np
import pandas as pd
import chardet 

In [3]:
file_path = 'IRENA_RenewableEnergy_Statistics_2000-2022.csv'

with open(file_path, 'rb') as f:
    result = chardet.detect(f.read())

df_irena = pd.read_csv(file_path, encoding=result['encoding'])

file_path_1 = 'organised_Gen.csv'

with open(file_path_1, 'rb') as f:
    result = chardet.detect(f.read())

df_us_data = pd.read_csv(file_path_1, encoding=result['encoding'])

file_path_2 = '02 modern-renewable-energy-consumption.csv'

with open(file_path_2, 'rb') as f:
    result = chardet.detect(f.read())

df_world_data = pd.read_csv(file_path_2, encoding=result['encoding'])

In [10]:
df_irena_clean = df_irena.dropna(subset=["Electricity Generation (GWh)"]).copy()

# Filter for valid records with known technology
df_subset = df_irena_clean[df_irena_clean["Technology"] != "Total"]

# Encode categorical variables
df_encoded = pd.get_dummies(df_subset[["Country", "Technology", "Year"]])
X = df_encoded
y = df_subset["Electricity Generation (GWh)"].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Parameter grid
param_grid = {
    'n_estimators': [50, 100, 150],
    'learning_rate': [0.05, 0.1, 0.15],
    'max_depth': [3, 5, 7]
}

# GridSearchCV for hyperparameter tuning
gbr = GradientBoostingRegressor(random_state=42)
grid_search = GridSearchCV(estimator=gbr, param_grid=param_grid, cv=3, scoring='r2', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Evaluate best model
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

# Display best model parameters
results_df = pd.DataFrame({
    "Best Parameters": [str(grid_search.best_params_)],
    "Mean Squared Error": [mse],
    "R-squared": [r2]
})

grid_search_results_df = pd.DataFrame(grid_search.cv_results_)
grid_search_results_df

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_learning_rate,param_max_depth,param_n_estimators,params,split0_test_score,split1_test_score,split2_test_score,mean_test_score,std_test_score,rank_test_score
0,2.295687,0.036177,0.019158,0.002456,0.05,3,50,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",0.806181,0.717291,0.813897,0.779123,0.043835,27
1,4.590393,0.134529,0.035996,0.009044,0.05,3,100,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",0.857137,0.808082,0.871924,0.845714,0.027287,26
2,6.481307,0.072479,0.019990,0.003744,0.05,3,150,"{'learning_rate': 0.05, 'max_depth': 3, 'n_est...",0.872613,0.849397,0.887223,0.869745,0.015575,24
3,3.430372,0.121952,0.024474,0.013733,0.05,5,50,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est...",0.923055,0.898340,0.926409,0.915935,0.012516,18
4,6.255212,0.364653,0.027202,0.011371,0.05,5,100,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est...",0.952899,0.930282,0.953775,0.945652,0.010874,17
5,9.912658,0.713288,0.058028,0.025905,0.05,5,150,"{'learning_rate': 0.05, 'max_depth': 5, 'n_est...",0.960155,0.940844,0.962400,0.954466,0.009676,14
6,4.343630,0.223172,0.019721,0.001981,0.05,7,50,"{'learning_rate': 0.05, 'max_depth': 7, 'n_est...",0.948074,0.939771,0.957136,0.948327,0.007092,15
7,9.050669,0.568056,0.030251,0.006813,0.05,7,100,"{'learning_rate': 0.05, 'max_depth': 7, 'n_est...",0.966037,0.958977,0.972012,0.965675,0.005328,10
8,12.105021,1.548657,0.041887,0.012892,0.05,7,150,"{'learning_rate': 0.05, 'max_depth': 7, 'n_est...",0.969699,0.963532,0.975504,0.969578,0.004888,6
9,1.654470,0.344626,0.009340,0.000365,0.10,3,50,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.866901,0.817306,0.878644,0.854284,0.026583,25


In [11]:
results_df

,Best Parameters,Mean Squared Error,R-squared
0,"{'learning_rate': 0.15, 'max_depth': 7, 'n_est...",3.646364e+08,0.983799


In [12]:
best_model

GradientBoostingRegressor(learning_rate=0.15, max_depth=7, n_estimators=150,
                          random_state=42)